# Step 4d — Feature Audit: Significance and Redundancy

**Question.** Before the engineered features go into a neural ranker, do they each contribute significantly to the ranking, and how much do they overlap?

**Why not p-values from a regression?** With 15.5M rows every coefficient is "significant", and ranking quality is measured per slate, not per row. The audit therefore asks the question that matters for the task: **does ranking quality on unseen users drop when a feature's information is destroyed?**

| Analysis | Method | Answers |
|---|---|---|
| Redundancy | Spearman correlation between features (monotone, robust to heavy tails), then hierarchical clustering into groups with average \|ρ\| ≥ 0.7 | Which features carry overlapping information |
| Significance | **Permutation importance** on the test slates: shuffle one feature's values, re-score, and measure the drop in NDCG@10 per slate. With ~25,800 test slates, the mean drop has a tight normal confidence interval; significance uses a **Bonferroni-corrected** one-sided test across all features | Whether each feature contributes beyond chance |
| Overlap-aware significance | Permute each **correlated group** jointly | Features that look useless alone because a near-duplicate substitutes for them |

Gain share (how often LightGBM split on a feature) is shown for reference only. It measures use, not contribution.

**Inputs:** `features.parquet` (55 features after Step 4b; Step 4c's seasonal columns are excluded if present). **Runtime:** ~25 min on Colab High-RAM CPU.

In [ ]:
import os, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.dataset as ds
from scipy.stats import norm
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'
SEED = 42
USER_FRAC = 1.0           # share of train/val users used to fit the audit model; test slates are always all used
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']

dset = ds.dataset(OUT_DIR / 'features.parquet')
FEATURES = [c for c in dset.schema.names if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
keep_user = np.random.default_rng(SEED + 1).random(200_000) < USER_FRAC   # not SEED: Step 2 drew the split from that same stream

def load(split, frac_filter=True):
    d = dset.to_table(columns=['user', 'label'] + FEATURES, filter=ds.field('split') == split).to_pandas()
    if frac_filter and USER_FRAC < 1:
        d = d[keep_user[d['user'].to_numpy()]]
    d = d.iloc[np.lexsort((np.random.default_rng(SEED).random(len(d)), d['user'].to_numpy()))]   # shuffle within slate
    return d[d.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)

tr, va, te = load('train'), load('val'), load('test', frac_filter=False)
print(f'{len(FEATURES)} features | train {tr["user"].nunique():,} slates | val {va["user"].nunique():,} | '
      f'test {te["user"].nunique():,} (all)')

## 1. Redundancy: which features overlap?

Spearman correlation on a 400k-row sample of the training slates. Features are clustered by average linkage on the distance 1 − |ρ| and cut at 0.3, so each group's members correlate at |ρ| ≥ 0.7 on average.

In [ ]:
smp = tr.sample(min(400_000, len(tr)), random_state=SEED)[FEATURES]
arr = smp.rank().corr().fillna(0.0).to_numpy(copy=True)
np.fill_diagonal(arr, 1.0)
rho = pd.DataFrame(arr, index=FEATURES, columns=FEATURES)
dist = np.clip(1 - np.abs(arr), 0, None)
np.fill_diagonal(dist, 0.0)
Z = linkage(squareform(dist, checks=False), method='average')
cluster_of = pd.Series(fcluster(Z, t=0.3, criterion='distance'), index=FEATURES)
order = [FEATURES[i] for i in leaves_list(Z)]

fig, ax = plt.subplots(figsize=(12, 10.5))
im = ax.imshow(rho.loc[order, order].abs(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7)
ax.set_title('|Spearman ρ| between engineered features (clustered)')
fig.colorbar(im, ax=ax, shrink=0.6); plt.tight_layout(); plt.show()

pairs = [(a, b, rho.loc[a, b]) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(rho.loc[a, b]) >= 0.8]
print(f'{len(pairs)} feature pairs with |ρ| >= 0.8:')
for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
    print(f'  {r:+.3f}  {a}  ~  {b}')
groups = {k: list(v.index) for k, v in cluster_of.groupby(cluster_of) if len(v) > 1}
print(f'\n{len(groups)} correlated groups (|ρ| >= 0.7 on average), covering '
      f'{sum(map(len, groups.values()))} of {len(FEATURES)} features:')
for k, members in groups.items():
    print(f'  group {k}: {members}')

## 2. Fit the reference model

Same LightGBM LambdaRank setup as Steps 2–4b (seed, shuffled slates, early stopping on validation NDCG@10), trained on all audited features.

In [ ]:
import lightgbm as lgb, warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')
warnings.filterwarnings('ignore', message='.*does not have valid feature names.*')
g_tr = tr.groupby('user', sort=False).size().to_numpy(); g_va = va.groupby('user', sort=False).size().to_numpy()
t0 = time.time()
model = lgb.LGBMRanker(objective='lambdarank', n_estimators=600, learning_rate=0.05, num_leaves=63,
                       min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, verbose=-1)
model.fit(tr[FEATURES], tr['label'], group=g_tr, eval_set=[(va[FEATURES], va['label'])], eval_group=[g_va],
          eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)])
del tr, va; gc.collect()
print(f'trained in {time.time() - t0:.0f}s, best iteration {model.best_iteration_}')

def slate_ndcg(u, lab, score, k=10, seed=0):
    """NDCG@10 of every slate, in a fixed slate order (sorted by user), with random tie-breaking."""
    r = np.random.default_rng(seed).random(len(u))
    order = np.lexsort((r, -score, u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    disc = np.where(rank < k, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), size)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    lab_i = lab[np.lexsort((-lab, u))]
    idcg = np.bincount(gid, weights=(2 ** lab_i - 1) * disc)
    return dcg / idcg

u_te, lab_te = te['user'].to_numpy(), te['label'].to_numpy().astype(np.float64)
X = te[FEATURES].copy(); del te; gc.collect()
base = slate_ndcg(u_te, lab_te, model.predict(X))
print(f'reference model test NDCG@10 = {base.mean():.4f} over {len(base):,} slates')

## 3. Permutation importance with confidence intervals

For each feature (and then each correlated group, and each feature family), its values are shuffled across all test rows with one fixed permutation, and every slate is re-scored. The per-slate drop in NDCG@10 is averaged over ~25,800 slates, which gives a normal-approximation 95% CI. A feature counts as **significant** when its mean drop exceeds zero at a one-sided α = 0.05 after Bonferroni correction across all individual features.

In [ ]:
rng = np.random.default_rng(SEED)
m_tests = len(FEATURES)
z_crit = norm.ppf(1 - 0.05 / m_tests)

def perm_drop(cols):
    idx = rng.permutation(len(X))
    saved = X[cols].to_numpy().copy()
    X[cols] = saved[idx]
    d = base - slate_ndcg(u_te, lab_te, model.predict(X))
    X[cols] = saved
    se = d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), d.mean() - 1.96 * se, d.mean() + 1.96 * se, d.mean() / se if se > 0 else 0.0

t0 = time.time()
rows = {f: perm_drop([f]) for f in FEATURES}
print(f'individual permutations: {time.time() - t0:.0f}s')
gain = pd.Series(model.booster_.feature_importance('gain'), index=FEATURES); gain /= gain.sum()
res = pd.DataFrame(rows, index=['ndcg_drop', 'ci_low', 'ci_high', 'z']).T
res['significant'] = res['z'] > z_crit
res['gain_share'] = gain
res['group'] = cluster_of.where(cluster_of.map(cluster_of.value_counts()) > 1)
print(f'Bonferroni threshold: z > {z_crit:.2f} (one-sided, {m_tests} tests)')
res.sort_values('ndcg_drop', ascending=False).round({'ndcg_drop': 5, 'ci_low': 5, 'ci_high': 5, 'z': 1, 'gain_share': 4})

In [ ]:
# overlap-aware: permute each correlated group jointly (one shared permutation keeps the group's internal correlation)
grp_rows = {}
for k, members in groups.items():
    dm, lo, hi, z = perm_drop(members)
    grp_rows[f'group {k} ({len(members)})'] = {'ndcg_drop': dm, 'ci_low': lo, 'ci_high': hi, 'z': z,
        'sum of individual drops': res.loc[members, 'ndcg_drop'].sum(),
        'members significant alone': int(res.loc[members, 'significant'].sum()), 'members': ', '.join(members)}
grp = pd.DataFrame(grp_rows).T if grp_rows else pd.DataFrame(columns=['ndcg_drop', 'z', 'members'])
grp['significant'] = grp['z'].astype(float) > z_crit
print('Joint drop >> sum of individual drops means the members substitute for each other (redundant).')
grp.sort_values('ndcg_drop', ascending=False)

In [ ]:
FAMILY = [('user x item', ('uc_',)), ('user x brand/category', ('ubrand_', 'ucat')),
          ('item popularity', ('i_users_',)), ('category / slate', ('c3_', 'slate_', 'cat3_')),
          ('query relevance', ('q_', 'qc_')), ('semantic personalisation', ('u_sem_',)),
          ('user-level (constant within slate)', ('u_',))]
def family(f):
    for name, prefixes in FAMILY:
        if f.startswith(prefixes):
            return name
    return 'other'
fam_of = pd.Series({f: family(f) for f in FEATURES})
fam_rows = {}
for name in fam_of.unique():
    members = list(fam_of.index[fam_of == name])
    dm, lo, hi, z = perm_drop(members)
    fam_rows[name] = {'features': len(members), 'ndcg_drop': dm, 'ci_low': lo, 'ci_high': hi, 'z': z}
pd.DataFrame(fam_rows).T.sort_values('ndcg_drop', ascending=False).round(5)

## 4. Verdict per feature

- **keep: significant** — significant alone.
- **keep one per group** — not significant alone, but its correlated group is: its information is real but duplicated. Keep the group's strongest member as the representative.
- **drop candidate** — not significant alone *and* not in a significant group: no detectable contribution on unseen users.

User-level features are a special case: they are constant within a slate, so a tree ranker can only use them through interactions. A neural model can combine them with history and candidate features, so they are flagged but kept for Step 5 unless they are near-duplicates.

In [ ]:
sig_groups = {int(str(k).split()[1]): m for k, m in zip(grp.index, grp['members'].str.split(', '))
              if grp.loc[k, 'significant']} if len(grp) else {}
rep = {k: res.loc[m, 'ndcg_drop'].idxmax() for k, m in sig_groups.items()}
def verdict(f):
    if res.loc[f, 'significant']:
        return 'keep: significant'
    g = res.loc[f, 'group']
    if pd.notna(g) and int(g) in sig_groups:
        return 'keep: group representative' if rep[int(g)] == f else 'redundant (group covered by ' + rep[int(g)] + ')'
    if fam_of[f].startswith('user-level'):
        return 'keep for neural model (user-level)'
    return 'drop candidate'
res['family'] = fam_of
res['verdict'] = [verdict(f) for f in res.index]
print(res['verdict'].str.replace(r' \(group covered by .*\)', ' (covered by group)', regex=True).value_counts().to_string())
res.sort_values(['verdict', 'ndcg_drop'], ascending=[True, False])[
    ['family', 'ndcg_drop', 'ci_low', 'ci_high', 'z', 'gain_share', 'group', 'verdict']].round(
    {'ndcg_drop': 5, 'ci_low': 5, 'ci_high': 5, 'z': 1, 'gain_share': 4})

In [ ]:
res.to_csv(OUT_DIR / 'feature_audit.csv')
print('saved', OUT_DIR / 'feature_audit.csv', '(Step 5 reads the verdicts from here)')
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

## 5. Takeaways

*(filled in after the run)*